# LEXICON-X | Cuaderno de investigación

**Objetivo:** comparar un baseline de recuperación con un método propuesto por el equipo. Compatible con Colab y Jupyter local.

Suban la carpeta `datos/` a `/content/` en Colab o ejecútenlo desde la carpeta `estudiantes/`. No contiene respuestas ocultas.

In [ ]:
from pathlib import Path
import csv, random, re, math, collections, unicodedata
random.seed(20261014)
BASE = Path('/content/datos') if Path('/content/datos/train_parallel.csv').exists() else Path('datos')
assert (BASE/'train_parallel.csv').exists(), 'Sube la carpeta datos a /content/ en Colab.'
def load_csv(name):
    with (BASE/name).open(encoding='utf-8',newline='') as f: return list(csv.DictReader(f))
train=load_csv('train_parallel.csv'); val=load_csv('validation_parallel.csv')
mono=load_csv('corpus_monolingual.csv'); public=load_csv('test_public_unlabeled.csv')
print('train',len(train),'valid',len(val),'monolingual',len(mono),'test público',len(public))
print('Ejemplo:',train[0])

## 1. Exploración e hipótesis
Anoten al menos dos hipótesis falsables: orden de palabras, morfología, negación o composición. Investiguen frecuencias y longitud de oraciones.

In [ ]:
def toks(s): return re.findall(r"[a-záéíóúñ]+",s.lower())
frequency=collections.Counter(t for r in mono for t in toks(r['naviri']))
print('Tokens frecuentes:',frequency.most_common(20))
print('Longitudes:',collections.Counter(len(toks(r['naviri'])) for r in mono))
# TODO: estudiar sufijos, bigramas, coocurrencia y patrones por posición.

## 2. Baseline obligatorio: recuperación por similitud de Jaccard
El baseline copia la traducción del ejemplo de entrenamiento que comparte más tokens con la entrada. Es intencionalmente simple: permite medir cuánto aporta el método propuesto.

In [ ]:
def jaccard(a,b):
    a,b=set(toks(a)),set(toks(b))
    return len(a&b)/len(a|b) if a|b else 0.0

def baseline_translate(sentence):
    best=max(train,key=lambda r:jaccard(sentence,r['naviri']))
    return best['espanol']

def norm(s):
    s=s.lower().strip()
    s=re.sub(r'[^\w\sáéíóúñü]','',s)
    return ' '.join(s.split())

def exact_match(rows,translator):
    return sum(norm(translator(r['naviri']))==norm(r['espanol']) for r in rows)/len(rows)
print('Baseline EM validación:',round(exact_match(val,baseline_translate),4))

## 3. Tu propuesta de investigación
Implementa un traductor propio. Puede ser: reglas inducidas de datos, modelos n-gram, representaciones vectoriales, Transformer preentrenado, seq2seq pequeño o sistema híbrido. No está permitido leer el solucionario docente. **Justifica por qué el método debería generalizar.**

In [ ]:
def my_translate(sentence):
    # TODO: reemplazar por el sistema propuesto.
    return baseline_translate(sentence)
print('Mi sistema EM validación:',round(exact_match(val,my_translate),4))

## 4. Comparación y análisis de errores
Incluye al menos una ablación (p. ej. sin datos monolingües, sin normalización, sin reglas de sufijos). Explica tres errores concretos.

In [ ]:
errors=[]
for r in val:
    pred=my_translate(r['naviri'])
    if norm(pred)!=norm(r['espanol']):
        errors.append((r['naviri'],r['espanol'],pred))
for source,ref,pred in errors[:5]:
    print('NÁVIRI:',source,'\nREF:',ref,'\nPRED:',pred,'\n')
print('Errores:',len(errors),'/',len(val))
# OPCIONAL: !pip -q install sacrebleu
# from sacrebleu.metrics import CHRF
# score=CHRF().corpus_score([my_translate(r['naviri']) for r in val],[[r['espanol'] for r in val]])
# print(score)

## 5. Reto sorpresa: exportar predicciones
Genera un archivo `predicciones.csv` con `id,prediccion`. En Colab se podrá descargar desde el explorador de archivos.

In [ ]:
out=Path('/content/predicciones.csv') if Path('/content').exists() else Path('predicciones.csv')
with out.open('w',encoding='utf-8',newline='') as f:
    w=csv.DictWriter(f,fieldnames=['id','prediccion']);w.writeheader()
    for r in public:w.writerow({'id':r['id'],'prediccion':my_translate(r['naviri'])})
print('Guardado:',out.resolve())

## 6. Conclusión de investigación
Completa: hipótesis, evidencia cuantitativa, ablación, casos fallidos, amenazas a la validez y siguiente experimento. Declara cualquier uso de IA externa.